# ARC-AGI-3 — Step 2: Action-Effect Tracking

Builds on Step 1 (available_actions-aware random agent). Adds the core idea behind the official preview competition's 1st-place entry (StochasticGoose, Tufa Labs): learn which actions actually change the game state, and prefer those over ones that are known no-ops -- instead of a trained CNN, we use simple per-action change-rate counting, which is transparent and easy to reason about.

Core loop each turn:
1. Compare the current frame to the frame before our last action -> did it change?
2. Record that outcome against the action (or screen region, for the coordinate    action ACTION6) we just took.
3. Next turn, prefer actions/regions with a higher observed change-rate. Untried    actions start optimistic (score 1.0) so the agent still explores everything at    least once before favoring anything.

In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv


In [ ]:
%%writefile /kaggle/working/my_agent.py
import hashlib
import random
import time
from typing import Any, Dict, List, Optional, Tuple

from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent

REGION_SIZE = 8  # bucket the 64x64 coordinate grid into 8x8 regions


class MyAgent(Agent):
    """Tracks, per action (and per screen region for the coordinate action),
    how often taking it actually changes the frame. Prefers actions/regions
    with a higher observed change-rate; untried ones start optimistic so
    everything gets tried before anything is written off.
    """

    MAX_ACTIONS = 80
    EPSILON = 0.15  # fraction of turns we pick uniformly at random, to keep exploring

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        seed = int(time.time() * 1000000) + hash(self.game_id) % 1000000
        random.seed(seed)
        # action_id -> [times_tried, times_it_changed_the_frame]
        self.action_stats: Dict[int, List[int]] = {}
        # (region_x, region_y) -> [times_tried, times_it_changed_the_frame]
        self.region_stats: Dict[Tuple[int, int], List[int]] = {}
        self.last_hash: Optional[str] = None
        self.pending_id: Optional[int] = None
        self.pending_region: Optional[Tuple[int, int]] = None

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    def _frame_hash(self, latest_frame: FrameData) -> str:
        return hashlib.md5(str(latest_frame.frame).encode()).hexdigest()

    def _record_outcome(self, current_hash: str) -> None:
        """Attribute whether the frame changed to whatever we did last turn."""
        if self.pending_id is None or self.last_hash is None:
            return
        changed = current_hash != self.last_hash
        stats = self.action_stats.setdefault(self.pending_id, [0, 0])
        stats[0] += 1
        stats[1] += int(changed)
        if self.pending_region is not None:
            rstats = self.region_stats.setdefault(self.pending_region, [0, 0])
            rstats[0] += 1
            rstats[1] += int(changed)

    def _score(self, stats_dict: dict, key: Any) -> float:
        tried, changed = stats_dict.get(key, [0, 0])
        if tried == 0:
            return 1.0  # optimistic: never-tried options get first pick
        return changed / tried

    def _pick_best(self, stats_dict: dict, keys: list) -> Any:
        """Epsilon-greedy: usually take the highest change-rate option, with
        ties broken randomly; sometimes (EPSILON) just explore at random."""
        if random.random() < self.EPSILON:
            return random.choice(keys)
        best_score = max(self._score(stats_dict, k) for k in keys)
        best_keys = [k for k in keys if self._score(stats_dict, k) == best_score]
        return random.choice(best_keys)

    def choose_action(
        self, frames: list[FrameData], latest_frame: FrameData
    ) -> GameAction:
        if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            self.last_hash = None
            self.pending_id = None
            self.pending_region = None
            return GameAction.RESET

        current_hash = self._frame_hash(latest_frame)
        self._record_outcome(current_hash)

        legal_ids = [a for a in latest_frame.available_actions if a != GameAction.RESET.value]
        if not legal_ids:
            legal_ids = [a.value for a in GameAction if a is not GameAction.RESET]

        chosen_id = self._pick_best(self.action_stats, legal_ids)
        action = GameAction.from_id(chosen_id)
        self.pending_id = chosen_id

        if action.is_simple():
            self.pending_region = None
            tried, changed = self.action_stats.get(chosen_id, [0, 0])
            action.reasoning = f"id={chosen_id} change_rate={changed}/{tried}"
        elif action.is_complex():
            n_regions = 64 // REGION_SIZE
            region_keys = [(rx, ry) for rx in range(n_regions) for ry in range(n_regions)]
            region = self._pick_best(self.region_stats, region_keys)
            self.pending_region = region
            x = region[0] * REGION_SIZE + random.randint(0, REGION_SIZE - 1)
            y = region[1] * REGION_SIZE + random.randint(0, REGION_SIZE - 1)
            action.set_data({"x": x, "y": y})
            tried, changed = self.region_stats.get(region, [0, 0])
            action.reasoning = {
                "region": str(region),
                "change_rate": f"{changed}/{tried}",
            }

        self.last_hash = current_hash
        return action


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    \"random\": Random,
    \"myagent\": MyAgent,
}
""")

    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent


In [ ]:
import os
import pandas as pd

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
